In [28]:
import pandas as pd

url = "https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv"
df = pd.read_csv(url)

print(df.shape)
print(df['Loan_Status'].unique())

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



(491, 14)
[1 0]


In [30]:
df.drop("Unnamed: 0", axis=1 ,inplace = True)

In [31]:
# 1. Lazımsız sütunu sil
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

# 2. Target artıq 0/1-dir, sadəcə yoxla
print("Unique target values:", df['Loan_Status'].unique())
print("NaN in target:", df['Loan_Status'].isnull().sum())

# 3. Features və target
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# 4. Numeric və categorical sütunları ayır
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# 5. Preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 8. Train
model.fit(X_train, y_train)

# 9. Predict
y_pred = model.predict(X_test)

# 10. Evaluation
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 11. Sample prediction
sample = X_test.iloc[0:1]
prediction = model.predict(sample)

print("\nSample data:")
print(sample)
print("\nPrediction:", "Approved" if prediction[0] == 1 else "Not Approved")

Unique target values: [1 0]
NaN in target: 0

Accuracy: 0.8383838383838383

Confusion Matrix:
[[17 13]
 [ 3 66]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.57      0.68        30
           1       0.84      0.96      0.89        69

    accuracy                           0.84        99
   macro avg       0.84      0.76      0.79        99
weighted avg       0.84      0.84      0.83        99


Sample data:
   Gender Married Dependents Education Self_Employed  ApplicantIncome  \
20   Male     Yes          0  Graduate            No             2425   

    CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
20             2340.0       143.0             360.0             1.0   

   Property_Area  
20     Semiurban  

Prediction: Approved


In [32]:
# ===== USER INPUT PREDICTION =====


user_data = {
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1.0,
    'Property_Area': 'Urban'
}

user_df = pd.DataFrame([user_data])

# Predict
prediction = model.predict(user_df)

print("\n=== USER INPUT RESULT ===")
print("Prediction:", "Loan Approved ✅" if prediction[0] == 1 else "Loan Not Approved ❌")


=== USER INPUT RESULT ===
Prediction: Loan Approved ✅
